# Final Python Notebook 2: Classification Modelling & Hyperparameter Tuning
## 5DATA002W.2 Machine Learning & Data Mining — Case Study A
**Author:** [Your Name Here]  
**Code Peer Review conducted by:** [Peer Reviewer Name]  
**Date of Peer Review:** [Date]

## Section 1: Import Libraries
Code block leveraged and reused from: Seminar Session — Classification Lab

In [ ]:
# Import pandas for data manipulation
import pandas as pd

# Import numpy for numerical operations
import numpy as np

# Import matplotlib and seaborn for visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Import data preprocessing tools
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Import train-test split utility
from sklearn.model_selection import train_test_split, GridSearchCV

# Import classification algorithms
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

# Import evaluation metrics
from sklearn.metrics import (confusion_matrix, classification_report,
                              accuracy_score, recall_score, precision_score,
                              f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay)

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

## Section 2: Load and Preprocess Data
Code block leveraged and reused from: Seminar Session — Data Preparation Lab

In [ ]:
# Load the raw loan approval dataset from CSV
df = pd.read_csv('loan_approval_data__3_.csv')

# Drop the id column as it carries no predictive information
df_clean = df.drop(columns=['id'])

# Remove outlier rows where age exceeds 100 (data entry error)
df_clean = df_clean[df_clean['age'] <= 100]

# Remove outlier rows where employment length exceeds 50 years (implausible values)
df_clean = df_clean[df_clean['emplyment_length'] <= 50]

# Replace negative loan interest rate with NaN (recording error)
df_clean.loc[df_clean['loan_interest_rate'] < 0, 'loan_interest_rate'] = np.nan

# Replace negative max_allowed_loan values with NaN
df_clean.loc[df_clean['max_allowed_loan'] < 0, 'max_allowed_loan'] = np.nan

# Fill missing age values with median
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())

# Fill missing loan_interest_rate with median
df_clean['loan_interest_rate'] = df_clean['loan_interest_rate'].fillna(df_clean['loan_interest_rate'].median())

# Fill missing payment_default_on_file with mode
df_clean['payment_default_on_file'] = df_clean['payment_default_on_file'].fillna(df_clean['payment_default_on_file'].mode()[0])

# Fill missing max_allowed_loan with median
df_clean['max_allowed_loan'] = df_clean['max_allowed_loan'].fillna(df_clean['max_allowed_loan'].median())

# Initialise label encoders for categorical columns
le_home = LabelEncoder()
le_intent = LabelEncoder()
le_default = LabelEncoder()

# Encode home_ownership to numeric values
df_clean['home_ownership_enc'] = le_home.fit_transform(df_clean['home_ownership'])

# Encode loan_intent to numeric values
df_clean['loan_intent_enc'] = le_intent.fit_transform(df_clean['loan_intent'])

# Encode payment_default_on_file to numeric values
df_clean['payment_default_enc'] = le_default.fit_transform(df_clean['payment_default_on_file'])

print('Preprocessing complete. Dataset shape:', df_clean.shape)

## Section 3: Define Features and Target, Split Data
Code block leveraged and reused from: Seminar Session — Classification Lab

In [ ]:
# Define the list of input features to be used for classification modelling
# max_allowed_loan is excluded (it is the regression target, not a classification input)
clf_features = ['age', 'income', 'home_ownership_enc', 'emplyment_length',
                'loan_intent_enc', 'loan_amount', 'loan_interest_rate',
                'loan_income_ratio', 'payment_default_enc', 'credit_history_length']

# Create input feature matrix X and target vector y
X = df_clean[clf_features]
y = df_clean['loan_approval_status']

# Display feature names and data shape
print('Feature names:', X.columns.tolist())
print('X shape:', X.shape)
print('y shape:', y.shape)

In [ ]:
# Split data into training (80%) and test (20%) sets
# random_state=42 ensures reproducibility (same split every run)
# stratify=y ensures the class ratio of Approved:Rejected is preserved in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Display split sizes and verify class ratios
print('Train size:', X_train.shape, '| Test size:', X_test.shape)
print('\nTrain class ratio:')
print(y_train.value_counts(normalize=True))
print('\nTest class ratio:')
print(y_test.value_counts(normalize=True))

## Section 4: Feature Scaling
Code block leveraged and reused from: Seminar Session — Classification Lab

In [ ]:
# Initialise StandardScaler to standardise features (mean=0, std=1)
# Scaler is fitted on training data only to prevent data leakage
scaler = StandardScaler()

# Fit scaler on training data and transform training set
X_train_scaled = scaler.fit_transform(X_train)

# Apply same transformation to test set using training statistics
X_test_scaled = scaler.transform(X_test)

print('Feature scaling applied. X_train_scaled shape:', X_train_scaled.shape)

## Section 5: Build Classification Models — NB, LR, KNN
Code block leveraged and reused from: Seminar Session — Classification Lab

In [ ]:
# Initialise Naive Bayes classifier (Gaussian NB for continuous features)
nb_model = GaussianNB()

# Train NB model on scaled training data
nb_model.fit(X_train_scaled, y_train)

# Predict class labels on test set
y_pred_nb = nb_model.predict(X_test_scaled)

# Predict class probabilities on test set for AUC-ROC
y_prob_nb = nb_model.predict_proba(X_test_scaled)[:, 1]

print('NB model trained and predictions generated.')

In [ ]:
# Initialise Logistic Regression classifier
# max_iter=1000 ensures convergence; random_state=42 for reproducibility
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# Train LR model on scaled training data
lr_model.fit(X_train_scaled, y_train)

# Predict class labels on test set
y_pred_lr = lr_model.predict(X_test_scaled)

# Predict class probabilities on test set for AUC-ROC
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print('LR model trained and predictions generated.')

In [ ]:
# Initialise K-Nearest Neighbours classifier with k=5 neighbours
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train KNN model on scaled training data
knn_model.fit(X_train_scaled, y_train)

# Predict class labels on test set
y_pred_knn = knn_model.predict(X_test_scaled)

# Predict class probabilities on test set for AUC-ROC
y_prob_knn = knn_model.predict_proba(X_test_scaled)[:, 1]

print('KNN (k=5) model trained and predictions generated.')

## Section 6: Evaluate Models — Confusion Matrices, Reports and AUC-ROC
Code block leveraged and reused from: Seminar Session — Model Evaluation Lab

In [ ]:
# Define a helper function to display confusion matrix for a given model
def plot_confusion_matrix(y_test, y_pred, model_name):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Approved(0)', 'Rejected(1)'])
    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, cmap='Blues')
    ax.set_title(f'Confusion Matrix — {model_name}')
    plt.tight_layout()
    plt.show()

# Plot confusion matrix for Naive Bayes
plot_confusion_matrix(y_test, y_pred_nb, 'Naive Bayes (NB)')

In [ ]:
# Print classification report for Naive Bayes
print('=== Naive Bayes Classification Report ===')
print(classification_report(y_test, y_pred_nb, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# Plot confusion matrix for Logistic Regression
plot_confusion_matrix(y_test, y_pred_lr, 'Logistic Regression (LR)')

In [ ]:
# Print classification report for Logistic Regression
print('=== Logistic Regression Classification Report ===')
print(classification_report(y_test, y_pred_lr, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# Plot confusion matrix for KNN
plot_confusion_matrix(y_test, y_pred_knn, 'K-Nearest Neighbours (KNN, k=5)')

In [ ]:
# Print classification report for KNN
print('=== KNN (k=5) Classification Report ===')
print(classification_report(y_test, y_pred_knn, target_names=['Approved(0)', 'Rejected(1)']))

In [ ]:
# Plot AUC-ROC curves for all three models on the same graph
fig, ax = plt.subplots(figsize=(8, 6))

# Compute and plot ROC curve for each model
for y_prob, name in [(y_prob_nb, 'NB'), (y_prob_lr, 'LR'), (y_prob_knn, 'KNN (k=5)')]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})')

# Plot diagonal reference line (random classifier)
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('AUC-ROC Curves — NB, LR, KNN')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Print all five evaluation metric scores for all three models
models_summary = {
    'NB':  (y_pred_nb, y_prob_nb),
    'LR':  (y_pred_lr, y_prob_lr),
    'KNN (k=5)': (y_pred_knn, y_prob_knn)
}

print(f"{'Model':<15} {'Accuracy':>10} {'Recall':>10} {'Precision':>10} {'F1':>10} {'AUC-ROC':>10}")
print('-' * 65)
for name, (y_pred, y_prob) in models_summary.items():
    acc  = accuracy_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_prob)
    print(f"{name:<15} {acc:>10.4f} {rec:>10.4f} {prec:>10.4f} {f1:>10.4f} {auc:>10.4f}")

## Section 7: Hyperparameter Tuning — GridSearchCV on Best Model (KNN)
Code block leveraged and reused from: Seminar Session — Hyperparameter Tuning Lab

In [ ]:
# Define hyperparameter grid for KNN tuning
# Testing different values of k (n_neighbors), weighting strategies, and distance metrics
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# Apply GridSearchCV to KNN using 5-fold cross-validation, optimising for recall
# recall is chosen as it aligns with the success criteria (maximise rejection detection)
knn_gs = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5, scoring='recall', n_jobs=-1)
knn_gs.fit(X_train_scaled, y_train)

# Display best hyperparameters found by GridSearchCV
print('Best hyperparameters:', knn_gs.best_params_)
print('Best cross-validated recall score:', knn_gs.best_score_)

In [ ]:
# Retrieve the tuned KNN model with best hyperparameters from GridSearchCV
knn_tuned = knn_gs.best_estimator_

# Generate predictions and probabilities on test set using the tuned model
y_pred_knn_tuned = knn_tuned.predict(X_test_scaled)
y_prob_knn_tuned = knn_tuned.predict_proba(X_test_scaled)[:, 1]

print('Tuned KNN predictions generated.')

In [ ]:
# Plot confusion matrix for tuned KNN model
plot_confusion_matrix(y_test, y_pred_knn_tuned, 'KNN Tuned (GridSearchCV)')

In [ ]:
# Print classification report for tuned KNN and compare with base KNN
print('=== Tuned KNN Classification Report ===')
print(classification_report(y_test, y_pred_knn_tuned, target_names=['Approved(0)', 'Rejected(1)']))

# Display metric comparison before and after tuning
print('\nMetric Comparison — KNN Before vs After Tuning:')
print(f"{'Metric':<12} {'Before':>10} {'After':>10}")
print('-' * 35)
for metric_name, before_val, after_val in [
    ('Recall',    recall_score(y_test, y_pred_knn),    recall_score(y_test, y_pred_knn_tuned)),
    ('Precision', precision_score(y_test, y_pred_knn), precision_score(y_test, y_pred_knn_tuned)),
    ('F1',        f1_score(y_test, y_pred_knn),        f1_score(y_test, y_pred_knn_tuned)),
    ('Accuracy',  accuracy_score(y_test, y_pred_knn),  accuracy_score(y_test, y_pred_knn_tuned)),
    ('AUC',       roc_auc_score(y_test, y_prob_knn),   roc_auc_score(y_test, y_prob_knn_tuned))
]:
    print(f"{metric_name:<12} {before_val:>10.4f} {after_val:>10.4f}")